# Computing q, k, v for ONE word

In [3]:
import numpy as np 

corpus = "the cat sat on the mat".split()
vocab = list(set(corpus))
w2i = {w: i for i, w in enumerate(vocab)}
i2w = {i: w for i, w in enumerate(vocab)}

E = np.random.randn(len(vocab), d)

In [ ]:
from scipy.special import softmax
d = 5 # dimension of my embedding for each word
dk = 3 # dimension of q/k/v - can differ from d, my choise

# The projection matrices are literally what changes the size — d is "before," dk is "after"
np.random.seed(42)
W_Q = np.random.randn(dk, d)
W_K = np.random.randn(dk, d)
W_V = np.random.randn(dk, d)

# test random x embedding to act as my input word embedding
x_1 = np.random.randn(d)

def project_qkv(x, W_Q, W_K, W_V):
    q = W_Q @ x 
    k = W_K @ x
    v = W_V @ x
    return q, k ,v

#q, k, v = project_qkv(x_1, W_Q, W_K, W_V)
#print(q, k ,v)


def project_sentence(words, E, w2i, W_Q, W_K, W_V):
    store = []
    for word in words:
        word_emb = E[w2i[word]]
        q, k, v = project_qkv(word_emb, W_Q, W_K, W_V)
        store.append([q, k, v])
    return store

store = project_sentence(corpus, E, w2i, W_Q, W_K, W_V)
#print(store)


def self_attention_scores(q_words, all_k):
    store_1 = []
    for q in q_words:
        store_2 = []
        for k in all_k:
            e_ij = np.dot(q, k)
            store_2.append(e_ij)
        store_1.append(store_2)
    return store_1 # each row represents each word q_i dot k_j score for all k in my corpus

all_q = [entry[0] for entry in store]
all_k = [entry[1] for entry in store]
all_v = [entry[2] for entry in store]

scores = self_attention_scores(all_q, all_k)
print(scores)


# lecture 5 slide 23 
def scale_scores(scores, dk):
    return np.array(scores)/np.sqrt(dk)

# helper functions used in parrallel with RNN
def attention_weights(scores):
    return softmax(scores)

def context_vector(weights, encoder_states):
    context = []
    for w, h in zip(weights, encoder_states):
        context.append(w * h)
    return sum(context)

# v = the value vector for one word (from project_qkv's third output)
def self_attention_output(scores, all_v, dk): 
    output = []
    for word_score in scores:
        scaled_score = scale_scores(word_score, dk)
        context_score = context_vector(attention_weights(scaled_score), all_v)
        output.append(context_score)
    return output

[[np.float64(0.11904357325772941), np.float64(1.0126082749008423), np.float64(-3.9689579539965343), np.float64(0.01888001438173006), np.float64(0.11904357325772941), np.float64(-3.013852990428033)], [np.float64(1.287489804089589), np.float64(0.17562556813077967), np.float64(-6.356360165768025), np.float64(0.3704780249726959), np.float64(1.287489804089589), np.float64(-3.878873336841957)], [np.float64(-3.4719142502228895), np.float64(6.734189993765424), np.float64(1.2807018580658223), np.float64(-1.1660708901108296), np.float64(-3.4719142502228895), np.float64(-2.6171428117282747)], [np.float64(-0.12621622164437885), np.float64(-3.25463349576022), np.float64(5.544305314975407), np.float64(0.05298884865124168), np.float64(-0.12621622164437885), np.float64(4.683753372818886)], [np.float64(0.11904357325772941), np.float64(1.0126082749008423), np.float64(-3.9689579539965343), np.float64(0.01888001438173006), np.float64(0.11904357325772941), np.float64(-3.013852990428033)], [np.float64(-3.96